In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM


config = PeftConfig.from_pretrained("ybelkada/opt-350m-lora")
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
lora_model = PeftModel.from_pretrained(model, "ybelkada/opt-350m-lora")

In [ ]:
!pip install -U trl

# Use TRL with PEFT (parameter effect fine tuning)

In [ ]:
from peft import LoraConfig
# this code will use for device_map="auto"
# model to a specific device using device_map={"": device_index}.



# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 6
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    peft_config=peft_config,  # LoRA configuration
    max_seq_length=max_seq_length,  # Maximum sequence length
    processing_class=tokenizer,
)

fine-tuned model with LoRA
Use the HuggingFaceTB/smoltalk dataset to fine-tune a deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B model

In [ ]:
!pip install -U transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback
from peft import LoraConfig
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch

# ── Device & Dataset ─────────────────────────────────────────────────────────
device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
print(device)

raw_dataset = load_dataset("HuggingFaceTB/smoltalk", "all")

# ── Model & Tokenizer ────────────────────────────────────────────────────────
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# ── LoRA Config ──────────────────────────────────────────────────────────────
peft_config = LoraConfig(
    r=6,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM"
)

In [ ]:
class LossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"Step: {state.global_step} | Loss: {logs['loss']:.4f} | Eval Loss: {logs.get('eval_loss', 'N/A')}")

# ── Training Config ──────────────────────────────────────────────────────────
args = SFTConfig(
    output_dir="./sft_output",
    max_steps=100,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=50,
    eval_strategy="steps",
    eval_steps=25,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# ── Trainer ──────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,         # ← pass raw base model, NOT get_peft_model(model)
    args=args,
    train_dataset=raw_dataset["train"],
    eval_dataset=raw_dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
    callbacks=[LossCallback()],
)

# Check trainable params after SFTTrainer wraps it
trainer.model.print_trainable_parameters()

trainer.train()

#Merging Implementation

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# 1. Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    "base_model_name", torch_dtype=torch.float16, device_map="auto"
)

# 2. Load the PEFT model with adapter
peft_model = PeftModel.from_pretrained(
    base_model, "path/to/adapter", torch_dtype=torch.float16
)

# 3. Merge adapter weights with base model
merged_model = peft_model.merge_and_unload()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("base_model_name")
merged_model.save_pretrained("path/to/save/merged_model")
tokenizer.save_pretrained("path/to/save/merged_model")